<a href="https://colab.research.google.com/github/Rosa-aa/flyrank-ml-internship/blob/main/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rosa-aa/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

My lane is Ranking / Scoring. The goal is to assign each content page a priority score based on its potential need for a refresh. This helps content editors identify which pages should be reviewed first. The model should rank pages using multiple performance, content, and keyword signals rather than simply classify them as declining or not declining.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Define the ML task type
ml_task_type = "Ranking / Scoring"

# Display the selected task type
print("My ML task:", ml_task_type)

My ML task: Ranking / Scoring


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

The intended target is future content performance decline, measured using a later time window. The model will use historical content and performance signals to rank pages by their risk of future decline. The current 30,000-row starter dataset does not contain this future outcome. Its is_declining_label is derived from a rule based on trend_direction, so it is only a rule-based proxy, not an independent observed target. A valid target requires a later performance window from the warehouse data.

In [ ]:
import pandas as pd

# Load the dataset
DATA_URL = "https://raw.githubusercontent.com/Rosa-aa/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_URL)

# Define the target and its source
target_name = "Future content performance decline"
target_type = "Future observed outcome"
target_available = False

# Check the current dataset label
proxy_column = "is_declining_label"

print("Target:", target_name)
print("Target type:", target_type)
print("Is future target available in current dataset?", target_available)
print("Available proxy:", proxy_column)

# Check whether the proxy column exists
print("Proxy column exists:", proxy_column in df.columns)

Target: Future content performance decline
Target type: Future observed outcome
Is future target available in current dataset? False
Available proxy: is_declining_label
Proxy column exists: False


## 3. Success metric

*One metric you can defend. What number means 'good'?*

I will use Precision@50. It measures the proportion of the 50 highest-ranked pages that actually experience the defined future decline. A good result should outperform a simple rule-based baseline on a client-held-out validation set. The exact target value will be set after calculating the baseline.

In [ ]:
# Define the success metric
success_metric = "Precision@50"
top_k = 50

# Define the evaluation goal
evaluation_goal = "Outperform the rule-based baseline"

print("Success metric:", success_metric)
print("Top K:", top_k)
print("Evaluation goal:", evaluation_goal)

Success metric: Precision@50
Top K: 50
Evaluation goal: Outperform the rule-based baseline


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

The unit of analysis is one content page. Each row represents one pseudonymized content item and contains its content properties, keyword context, and aggregated performance metrics. The dataset contains 30,000 rows across 32 pseudonymized clients.

In [ ]:
# Check the dataset structure
print("Dataset shape:", df.shape)

# Count unique content pages
print("Unique content pages:", df["content_id"].nunique())

# Count unique clients
print("Unique clients:", df["client_id"].nunique())

# Check duplicate content IDs
print("Duplicate content IDs:", df["content_id"].duplicated().sum())

# Display the first five rows
display(df.head())

Dataset shape: (30000, 44)
Unique content pages: 30000
Unique clients: 32
Duplicate content IDs: 0


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule may consider only one or two signals, such as page age or declining impressions. However, refresh priority can depend on several interacting factors, including impressions, clicks, CTR, engagement, content age, update history, keyword competition, and content type. These relationships may be nonlinear and differ across content types and clients. ML can learn patterns across multiple signals and produce a more flexible priority ranking. Its value must be demonstrated by outperforming a transparent rule-based baseline.

In [ ]:
# Define candidate features
candidate_features = [
    "search_volume",
    "competition",
    "cpc",
    "content_type",
    "main_intent",
    "word_count",
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "pageviews_90d",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

# Check which features are available
available_features = [
    col for col in candidate_features
    if col in df.columns
]

missing_features = [
    col for col in candidate_features
    if col not in df.columns
]

print("Available features:", len(available_features))
print("Missing features:", missing_features)

# Check missing values
missing_summary = (
    df[available_features]
    .isnull()
    .sum()
    .sort_values(ascending=False)
)

print("\nMissing values per feature:")
display(missing_summary)

Available features: 17
Missing features: []

Missing values per feature:


,0
word_count,7699
search_volume,2468
competition,2468
cpc,2468
main_intent,2374
scroll_rate,125
content_type,0
days_since_last_update,0
content_age_days,0
impressions_90d,0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.